RAW vs RAWH (k=10): cluster class-averaged concept fractions, attention-weighted RAWH, survival.
All figures saved without panel titles. Filenames are explicit.

In [ ]:


# Imports and config
from pathlib import Path
import gc, h5py, joblib, json
import numpy as np
import pandas as pd
import torch
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from models.model_clam import CLAM_MB
from scipy.stats import mannwhitneyu

# ---------------- Edit these paths ----------------
FEAT_DIR     = Path("/common/users/wq50/CLAM/features/HPV_UNI2_features/h5_files")
EMBED_DIM    = 1536
CLAM_WEIGHT  = Path("/common/users/wq50/CLAM/results/HPV_CLAM_50_mb_s1/s_9_checkpoint.pt")
RAW_MODEL    = Path("/common/users/wq50/CLAM2/kmeans_models/hpv_uni2_k10_raw.joblib")
RAWH_MODEL   = Path("/common/users/wq50/CLAM2/kmeans_models/hpv_uni2_k10_rawh.joblib")
LABELS_CSV   = Path("/common/users/wq50/CLAM/dataset_csv/HNSCC.csv")   # columns: case_id, slide_id, hpv_status, survival
LABEL_COL    = ""                            # set to force a specific HPV column
OUT_DIR      = Path("k10_compare_all_plots"); OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- Runtime settings ----------------
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
BATCH        = 16_384
K_ASSERT     = 10
N_BOOT       = 2000

np.random.seed(0); torch.manual_seed(0)
plt.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300,
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "font.size": 11, "axes.labelsize": 11, "legend.fontsize": 10
})

# %% — CLAM + streaming helpers
@torch.inference_mode()
def load_clam(weight_path: Path, device: str, embed_dim: int) -> CLAM_MB:
    m = CLAM_MB(gate=True, size_arg="small", n_classes=2, embed_dim=embed_dim)
    sd = torch.load(weight_path, map_location=device)
    m.load_state_dict(sd, strict=False)
    return m.to(device).eval()

def iter_h5_blocks(h5_path: Path, batch: int):
    with h5py.File(h5_path, "r") as f:
        feats = f["features"]; n = feats.shape[0]
        for off in range(0, n, batch):
            yield off, feats[off:off+batch][:].astype(np.float32)

@torch.inference_mode()
def h_and_alpha(block_np: np.ndarray, clam: CLAM_MB, device: str):
    """
    Returns:
      h: (m,512) float32
      a: (m,) float32 attention; softmax per block; renorm to mean≈1
    """
    x = torch.from_numpy(block_np).to(device)
    A_raw, h = clam.attention_net(x)          # A_raw: (m,2) -> use first logit
    logits = A_raw[:, 0]
    a = torch.softmax(logits, dim=0).clamp_min(1e-12)
    a = a * (a.numel() / a.sum())
    return h.cpu().numpy().astype(np.float32), a.cpu().numpy().astype(np.float32)

# %% — Label utilities
def normalize_hpv(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip().str.lower()
    pos = s.isin(["1","hpv+","positive","pos","true","yes"])
    neg = s.isin(["0","hpv-","negative","neg","false","no"])
    out = pd.Series(index=s.index, dtype="float32")
    out[pos] = 1.0; out[neg] = 0.0
    rem = out.isna()
    if rem.any(): out[rem] = pd.to_numeric(s[rem], errors="coerce")
    return out.astype("float32")

def normalize_survival(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip().str.lower()
    alive = s.isin(["survived","alive","yes","1","true"])
    dead  = s.isin(["deceased","dead","no","0","false"])
    out = pd.Series(index=s.index, dtype="float32")
    out[alive] = 1.0; out[dead] = 0.0
    return out.astype("float32")

def find_hpv_col(df: pd.DataFrame, user_col: str="") -> str:
    if user_col and user_col in df.columns: return user_col
    cands = [c for c in df.columns if c.lower() in ("label__hpv","hpv","hpv_status","label_hpv","hpv_label")]
    return cands[0] if cands else (df.columns[2] if len(df.columns) >= 3 else None)

# %% — Load models, labels, CLAM
km_raw  = joblib.load(RAW_MODEL)
km_rawh = joblib.load(RAWH_MODEL)
assert getattr(km_raw, "n_clusters", None) == K_ASSERT, "RAW model must be k=10"
assert getattr(km_rawh, "n_clusters", None) == K_ASSERT, "RAWH model must be k=10"

clam = load_clam(CLAM_WEIGHT, DEVICE, EMBED_DIM)

df_lab = pd.read_csv(LABELS_CSV)
assert "slide_id" in df_lab.columns, "labels_csv must include 'slide_id'"
hpv_col = find_hpv_col(df_lab, LABEL_COL); assert hpv_col is not None, "HPV column not found"
y_hpv = normalize_hpv(df_lab[hpv_col])
y_surv = normalize_survival(df_lab["survival"]) if "survival" in df_lab.columns else pd.Series(index=df_lab.index, dtype="float32")

slide_ids = sorted([p.stem for p in FEAT_DIR.glob("*.h5") if p.stem in set(df_lab["slide_id"].astype(str))])

# Map slide_id -> labels
lab_map_hpv  = pd.Series(y_hpv.values, index=df_lab["slide_id"].astype(str))
lab_map_surv = pd.Series(y_surv.values, index=df_lab["slide_id"].astype(str))

# %% — Fractions computation (RAW, RAWH, RAWH-AW)
def fractions_raw(h5_path: Path, km) -> np.ndarray:
    K = km.n_clusters; counts = np.zeros(K, dtype=np.float64); total = 0
    for _, blk in iter_h5_blocks(h5_path, BATCH):
        labs = km.predict(blk)
        counts += np.bincount(labs, minlength=K).astype(np.float64)
        total  += blk.shape[0]
    return counts / max(1, total)

def fractions_rawh(h5_path: Path, km, clam: CLAM_MB) -> np.ndarray:
    K = km.n_clusters; counts = np.zeros(K, dtype=np.float64); total = 0
    for _, blk in iter_h5_blocks(h5_path, BATCH):
        H, _ = h_and_alpha(blk, clam, DEVICE)
        labs = km.predict(H)
        counts += np.bincount(labs, minlength=K).astype(np.float64)
        total  += H.shape[0]
    return counts / max(1, total)

def fractions_rawh_aw(h5_path: Path, km, clam: CLAM_MB) -> np.ndarray:
    K = km.n_clusters; wsum = np.zeros(K, dtype=np.float64); asum = 0.0
    for _, blk in iter_h5_blocks(h5_path, BATCH):
        H, a = h_and_alpha(blk, clam, DEVICE)
        labs = km.predict(H)
        wsum += np.bincount(labs, weights=a.astype(np.float64), minlength=K)
        asum += float(a.sum())
    return wsum / max(1e-12, asum)

rows_raw, rows_rawh, rows_aw = [], [], []
for sid in slide_ids:
    h5 = FEAT_DIR / f"{sid}.h5"
    if not h5.exists(): continue
    rows_raw.append( (sid, *fractions_raw(h5, km_raw)) )
    rows_rawh.append((sid, *fractions_rawh(h5, km_rawh, clam)) )
    rows_aw.append(  (sid, *fractions_rawh_aw(h5, km_rawh, clam)) )
    gc.collect()

cols = ["slide_id"] + [f"f{k}" for k in range(K_ASSERT)]
df_raw     = pd.DataFrame(rows_raw, columns=cols)
df_rawh    = pd.DataFrame(rows_rawh, columns=cols)
df_rawh_aw = pd.DataFrame(rows_aw, columns=cols)

# attach labels
for df in (df_raw, df_rawh, df_rawh_aw):
    df["hpv"] = df["slide_id"].map(lab_map_hpv)
    df["survival"] = df["slide_id"].map(lab_map_surv)

# save per-slide CSVs
df_raw.to_csv(OUT_DIR / f"fractions_RAW_{RAW_MODEL.stem}.csv", index=False)
df_rawh.to_csv(OUT_DIR / f"fractions_RAWH_{RAWH_MODEL.stem}.csv", index=False)
df_rawh_aw.to_csv(OUT_DIR / f"fractions_RAWH-AW_{RAWH_MODEL.stem}.csv", index=False)

# %% — Bootstrap CI utilities
rng = np.random.default_rng(0)

def bootstrap_mean_ci(x: np.ndarray, n_boot=N_BOOT, alpha=0.05):
    x = x[np.isfinite(x)]
    n = x.size
    if n == 0: return np.nan, np.nan, np.nan
    means = np.empty(n_boot, dtype=np.float64)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n); means[b] = x[idx].mean()
    means.sort()
    m = x.mean()
    lo = means[int(alpha/2*n_boot)]
    hi = means[int((1-alpha/2)*n_boot)-1]
    return float(m), float(lo), float(hi)

def summarize_by_binary(df: pd.DataFrame, label_col: str):
    pos = df[label_col]==1.0; neg = df[label_col]==0.0
    K = sum(c.startswith("f") for c in df.columns)
    s_pos, s_neg = [], []
    for k in range(K):
        s_pos.append(bootstrap_mean_ci(df.loc[pos, f"f{k}"].values))
        s_neg.append(bootstrap_mean_ci(df.loc[neg, f"f{k}"].values))
    return s_pos, s_neg

# NEW: summarize within a condition (e.g., within HPV+ only)
def summarize_by_binary_conditional(df: pd.DataFrame, label_col: str, cond_col: str, cond_val: float):
    sub = df[np.isfinite(df[cond_col]) & (df[cond_col] == cond_val)]
    if sub.empty:
        return None, None
    return summarize_by_binary(sub, label_col)

def delta_vector(df: pd.DataFrame, label_col: str):
    K = sum(c.startswith("f") for c in df.columns)
    d = []
    for k in range(K):
        p = df.loc[df[label_col]==1.0, f"f{k}"].values
        n = df.loc[df[label_col]==0.0, f"f{k}"].values
        if p.size==0 or n.size==0: d.append(np.nan)
        else: d.append(float(np.nanmean(p) - np.nanmean(n)))
    return np.array(d)

# %% — Plotting helpers (no titles)
def bar_with_ci(stats_pos, stats_neg, ylabel: str, out_path: Path, legend_labels=("Class 1","Class 0")):
    if stats_pos is None or stats_neg is None:
        return
    K = len(stats_pos); x = np.arange(K); w = 0.38
    p_mean = [t[0] for t in stats_pos]; n_mean = [t[0] for t in stats_neg]
    p_lo   = [t[1] for t in stats_pos]; p_hi = [t[2] for t in stats_pos]
    n_lo   = [t[1] for t in stats_neg]; n_hi = [t[2] for t in stats_neg]
    p_err = [np.array(p_mean)-np.array(p_lo), np.array(p_hi)-np.array(p_mean)]
    n_err = [np.array(n_mean)-np.array(n_lo), np.array(n_hi)-np.array(n_mean)]

    fig, ax = plt.subplots(figsize=(6.0, 3.2))
    ax.bar(x-w/2, p_mean, w, yerr=p_err, capsize=2, label=legend_labels[0], edgecolor="black", linewidth=0.6)
    ax.bar(x+w/2, n_mean, w, yerr=n_err, capsize=2, label=legend_labels[1], edgecolor="black", linewidth=0.6)
    ax.set_xticks(x); ax.set_xticklabels([f"C{k}" for k in range(K)])
    ax.set_ylabel(ylabel)
    ax.legend(frameon=False, ncol=2)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    fig.tight_layout()
    fig.savefig(out_path.with_suffix(".pdf"))
    fig.savefig(out_path.with_suffix(".png"))
    plt.close(fig)

def plot_delta(delta_vec, ylabel: str, out_path: Path):
    order = np.argsort(-np.nan_to_num(delta_vec, nan=-1e9))
    x = np.arange(delta_vec.size)
    fig, ax = plt.subplots(figsize=(6.0, 3.2))
    ax.bar(x, delta_vec[order], edgecolor="black", linewidth=0.6)
    ax.axhline(0, color="black", linewidth=0.6)
    ax.set_xticks(x); ax.set_xticklabels([f"C{k}" for k in order])
    ax.set_ylabel(ylabel)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    fig.tight_layout()
    fig.savefig(out_path.with_suffix(".pdf"))
    fig.savefig(out_path.with_suffix(".png"))
    plt.close(fig)

def jsd(p, q, eps=1e-12):
    p = np.clip(p, eps, 1.0); q = np.clip(q, eps, 1.0)
    p = p/p.sum(); q = q/q.sum()
    m = 0.5*(p+q)
    def kl(a,b): return float(np.sum(a*np.log(a/b)))
    return 0.5*kl(p,m) + 0.5*kl(q,m)

def plot_jsd_scatter(df_a: pd.DataFrame, df_b: pd.DataFrame, labels: pd.Series, out_path: Path):
    A = df_a.set_index("slide_id"); B = df_b.set_index("slide_id")
    common = A.index.intersection(B.index)
    vals = []
    for sid in common:
        p = A.loc[sid, [f"f{k}" for k in range(K_ASSERT)]].values.astype(float)
        q = B.loc[sid, [f"f{k}" for k in range(K_ASSERT)]].values.astype(float)
        vals.append((sid, jsd(p,q), float(labels.loc[sid]) if sid in labels.index else np.nan))
    d = pd.DataFrame(vals, columns=["slide_id","jsd","label"])
    d.to_csv(out_path.with_suffix(".csv"), index=False)

    fig, ax = plt.subplots(figsize=(4.0,3.2))
    x = d["label"].values
    jitter = np.random.default_rng(0).normal(scale=0.02, size=len(x))
    ax.scatter(x+jitter, d["jsd"].values, s=10, linewidths=0.3, edgecolors="black")
    ax.set_xticks([0,1]); ax.set_xticklabels(["HPV−","HPV+"])
    ax.set_ylabel("Jensen–Shannon distance")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    fig.tight_layout()
    fig.savefig(out_path.with_suffix(".pdf"))
    fig.savefig(out_path.with_suffix(".png"))
    plt.close(fig)

# %% — HPV plots
# RAW vs RAWH HPV class bars
hpv_pos_raw,  hpv_neg_raw  = summarize_by_binary(df_raw,     "hpv")
hpv_pos_rawh, hpv_neg_rawh = summarize_by_binary(df_rawh,    "hpv")
hpv_pos_aw,   hpv_neg_aw   = summarize_by_binary(df_rawh_aw, "hpv")

bar_with_ci(hpv_pos_raw,  hpv_neg_raw,  "Mean concept fraction",
            OUT_DIR / f"HPV_bar_RAW_{RAW_MODEL.stem}_vs_class", legend_labels=("HPV+","HPV−"))
bar_with_ci(hpv_pos_rawh, hpv_neg_rawh, "Mean concept fraction",
            OUT_DIR / f"HPV_bar_RAWH_{RAWH_MODEL.stem}_vs_class", legend_labels=("HPV+","HPV−"))
bar_with_ci(hpv_pos_aw,   hpv_neg_aw,   "Mean concept fraction",
            OUT_DIR / f"HPV_bar_RAWH-AW_{RAWH_MODEL.stem}_vs_class", legend_labels=("HPV+","HPV−"))

# Δ contrast bars
d_raw  = delta_vector(df_raw, "hpv")
d_rawh = delta_vector(df_rawh, "hpv")
d_aw   = delta_vector(df_rawh_aw, "hpv")

plot_delta(d_rawh - d_raw, "Δ(HPV+ − HPV−)", OUT_DIR / f"HPV_delta_RAWH_minus_RAW_{RAWH_MODEL.stem}_vs_{RAW_MODEL.stem}")
plot_delta(d_aw   - d_rawh,"Δ(HPV+ − HPV−)", OUT_DIR / f"HPV_delta_RAWH-AW_minus_RAWH_{RAWH_MODEL.stem}")

# Slide-level JSD between RAW and RAWH
plot_jsd_scatter(df_raw, df_rawh, lab_map_hpv,
                 OUT_DIR / f"HPV_slide_JSD_RAW_vs_RAWH_{RAW_MODEL.stem}_vs_{RAWH_MODEL.stem}")

# %% — Survival plots (overall + stratified by HPV)
def safe_survival_plot(df_feats: pd.DataFrame, tag: str, out_prefix: str):
    """Make overall survival plot; also HPV+ and HPV− stratified plots if both classes present."""
    if "survival" not in df_feats.columns or df_feats["survival"].isna().all():
        return

    # overall
    surv_alive, surv_dead = summarize_by_binary(df_feats, "survival")
    bar_with_ci(surv_alive, surv_dead, "Mean concept fraction",
                OUT_DIR / f"{out_prefix}_{tag}_Survived_vs_Deceased",
                legend_labels=("Survived","Deceased"))

    # HPV+ only
    stats_pos, stats_neg = summarize_by_binary_conditional(df_feats, "survival", "hpv", 1.0)
    bar_with_ci(stats_pos, stats_neg, "Mean concept fraction",
                OUT_DIR / f"{out_prefix}_{tag}_HPVpos_Survived_vs_Deceased",
                legend_labels=("Survived (HPV+)","Deceased (HPV+)"))

    # HPV− only
    stats_pos, stats_neg = summarize_by_binary_conditional(df_feats, "survival", "hpv", 0.0)
    bar_with_ci(stats_pos, stats_neg, "Mean concept fraction",
                OUT_DIR / f"{out_prefix}_{tag}_HPVneg_Survived_vs_Deceased",
                legend_labels=("Survived (HPV−)","Deceased (HPV−)"))

# RAWH-AW (attention-weighted) — overall + per-HPV
safe_survival_plot(df_rawh_aw, RAWH_MODEL.stem, "SURV_bar_RAWH-AW")
# RAW
safe_survival_plot(df_raw, RAW_MODEL.stem, "SURV_bar_RAW")
# RAWH (unweighted)
safe_survival_plot(df_rawh, RAWH_MODEL.stem, "SURV_bar_RAWH")

# Per-cluster Mann–Whitney p-values (overall, RAWH-AW)
if "survival" in df_rawh_aw.columns and df_rawh_aw["survival"].notna().any():
    K = K_ASSERT
    pvals = []
    for k in range(K):
        a = df_rawh_aw.loc[df_rawh_aw["survival"]==1.0, f"f{k}"].values
        d = df_rawh_aw.loc[df_rawh_aw["survival"]==0.0, f"f{k}"].values
        if len(a) > 3 and len(d) > 3:
            _, p = mannwhitneyu(a, d, alternative="two-sided")
        else:
            p = np.nan
        pvals.append(p)
    pd.Series(pvals, index=[f"C{k}" for k in range(K)]).to_csv(
        OUT_DIR / f"SURV_pvals_RAWH-AW_{RAWH_MODEL.stem}.csv"
    )

# %% — Manifest of outputs
manifest = {
    "raw_model": str(RAW_MODEL),
    "rawh_model": str(RAWH_MODEL),
    "clam_weight": str(CLAM_WEIGHT),
    "embed_dim": EMBED_DIM,
    "outputs": sorted([str(p) for p in OUT_DIR.glob("*")])
}
with open(OUT_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("Done. See:", OUT_DIR)

Done. See: k10_compare_all_plots
